v1과 동일한 구조 그러나 텐스플로우 기반

## 구조설계
1. input:(H,W,C)  #H세로 픽셀수 W가로 픽셀수 C 채널수(색깔정보)
2. Block1: Con ->ReLU -> MaxPool #Con(이미지특징) ReLU(비선형) MaxPool (크기 줄이고 핵심정보 추출)
3. Block2: Con ->ReLU -> MaxPool
4. Block3: Con ->ReLU -> MaxPool
5. Head: GAP(or Flatten) → Dense → Output
6. Input shape
7. Output 방식 확정
8. Overfitting 방지

파일 가져오기

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
folder = "/content/drive/MyDrive/DS팀 데이터 저장소/"
print(os.listdir(folder)[:50])

['ORAL CANCER DATASET', 'Oral Images Dataset', 'Oral Cancer Images for Classification', '데이터전처리_정현.ipynb', 'Baseline_CNN_v2.ipynb', '원본데이터전처리_정현.ipynb', '원본데이터전처리_정현.txt', 'Baseline_CNN_v1.ipynb']


import 확인 GPU 체크

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TF version: 2.19.0
GPU: []


데이터 shape/dtype 체크

In [ ]:
print("X_train:", X_train.shape, X_train.dtype, "min/max:", X_train.min(), X_train.max())
print("y_train:", y_train.shape, y_train.dtype, "unique:", np.unique(y_train)[:20])

print("X_val:", X_val.shape, X_val.dtype)
print("y_val:", y_val.shape, y_val.dtype)

NameError: name 'X_train' is not defined

입 출력 설정 세팅 체크

In [ ]:
input_shape = X_train.shape[1:]  # (H, W, C)

unique = np.unique(y_train)
if len(unique) == 2 and set(unique.tolist()) <= {0, 1}:
    mode = "binary"
    num_classes = 1
    loss = keras.losses.BinaryCrossentropy()
    metrics = [keras.metrics.BinaryAccuracy(name="acc"), keras.metrics.AUC(name="auc")]
else:
    mode = "multiclass"
    num_classes = len(unique)
    # y가 정수 라벨(0..K-1)이라고 가정 (대부분 이거)
    loss = keras.losses.SparseCategoricalCrossentropy()
    metrics = [keras.metrics.SparseCategoricalAccuracy(name="acc")]

print("input_shape:", input_shape)
print("mode:", mode, "| num_classes:", num_classes)

모델 체크
과적합 방지(구조는 v1과 같이 그대로)

In [ ]:
def build_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)

    # Block 1: Conv -> ReLU -> MaxPool
    x = layers.Conv2D(32, (3,3), padding="same", use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.15)(x)

    # Block 2
    x = layers.Conv2D(64, (3,3), padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.20)(x)

    # Block 3
    x = layers.Conv2D(128, (3,3), padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.25)(x)

    # Head: GAP -> Dense -> Output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.40)(x)

    if num_classes == 1:
        outputs = layers.Dense(1, activation="sigmoid")(x)
    else:
        outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs)

model = build_cnn(input_shape, num_classes)
model.summary()

컴파일

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=loss,
    metrics=metrics
)

콜백및 학습

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

검증셋 평가+예측

In [ ]:
print("Eval:", model.evaluate(X_val, y_val, verbose=0))

pred = model.predict(X_val[:8])
print("pred shape:", pred.shape)

if num_classes == 1:
    print("pred (prob):", pred.reshape(-1))
else:
    print("pred class:", pred.argmax(axis=1))